In [ ]:
import pyspark
print(pyspark.__version__)
import sys
print(sys.executable)
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("Assignment46") \
        .getOrCreate()

print(spark.version)


In [4]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType, IntegerType, FloatType)
from pyspark.sql.functions import col, when, sum
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
def profiling_dataframe(df, name):
    null_counts = [
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)for c in df.columns]
    df.select(null_counts).show()
    print(f"{name} row count: {df.count()}")

data_path = "/opt/airflow/data_day_46"

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("internal_notes", StringType(), True)
])

df_orders = spark.read.csv(
    f"{data_path}/orders.csv",
    header=True,
    schema=orders_schema
)
                 
df_orders.show(5)
df_orders.printSchema()
profiling_dataframe(df_orders,"orders")

In [ ]:
order_items_schema = StructType([
    StructField("order_item_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("discount_rate", DoubleType(), True)
])

df_order_items = spark.read.csv(
    "airflow/data_day_46/order_items.csv",
    header=True,
    schema=order_items_schema
)
                 
df_order_items.show(5)
df_order_items.printSchema()
profiling_dataframe(df_order_items,"order_items")

+-------------+---------+----------+----------------+--------+----------+-------------+
|order_item_id| order_id|product_id|product_category|quantity|     price|discount_rate|
+-------------+---------+----------+----------------+--------+----------+-------------+
|  ITEM-000001|ORD-00001| PROD-1213|           Books|       3|1373263.87|          0.2|
|  ITEM-000002|ORD-00001| PROD-1006| Health & Beauty|       1| 698654.31|          0.2|
|  ITEM-000003|ORD-00001| PROD-1319|     Electronics|       5|1193248.21|          0.0|
|  ITEM-000004|ORD-00001| PROD-1327|         Fashion|       2|2274478.57|          0.0|
|  ITEM-000005|ORD-00002| PROD-1160|      Automotive|       4|2293128.34|          0.1|
+-------------+---------+----------+----------------+--------+----------+-------------+
only showing top 5 rows

root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-

In [ ]:
df_order_summary = df_orders.join(df_order_items.select("order_id",
                                                        "product_category",
                                                        "quantity", 
                                                        "price",
                                                        "discount_rate"),
                                  on="order_id",
                                  how="left")

df_order_summary.show()

+---------+-----------+----------+------------+--------------+-------------------+----------------+--------+----------+-------------+
| order_id|customer_id|order_date|order_status|payment_method|     internal_notes|product_category|quantity|     price|discount_rate|
+---------+-----------+----------+------------+--------------+-------------------+----------------+--------+----------+-------------+
|ORD-00001|   CUST-229|2026-10-07|   COMPLETED|           OVO|               NULL|         Fashion|       2|2274478.57|          0.0|
|ORD-00001|   CUST-229|2026-10-07|   COMPLETED|           OVO|               NULL|     Electronics|       5|1193248.21|          0.0|
|ORD-00001|   CUST-229|2026-10-07|   COMPLETED|           OVO|               NULL| Health & Beauty|       1| 698654.31|          0.2|
|ORD-00001|   CUST-229|2026-10-07|   COMPLETED|           OVO|               NULL|           Books|       3|1373263.87|          0.2|
|ORD-00002|   CUST-169|2026-10-07|    REFUNDED|          NULL|

In [ ]:
df_order_summary = df_order_summary.drop("internal_notes").fillna("Unknown")
df_order_summary = df_order_summary.withColumn("order_status", F.initcap(F.trim("order_status"))).withColumn(
                                               "payment_method", F.initcap(F.trim("payment_method"))).withColumn(
                                               "product_category", F.initcap(F.trim("product_category")))

df_order_summary = df_order_summary.filter(F.col("order_status") == "Completed")
df_order_summary.show()

In [ ]:
df_order_summary = (
    df_order_summary
    .drop("payment_method", "product_category")
    .drop("product_category")
    .withColumn("total_price", F.round(F.col("quantity") * F.col("price"),2))
    .withColumn("total_after_discount", F.round(F.col("total_price") * (1 - F.col("discount_rate")),2))
)

df_order_summary.show()

+---------+-----------+----------+------------+--------+----------+-------------+------------+--------------------+
| order_id|customer_id|order_date|order_status|quantity|     price|discount_rate| total_price|total_after_discount|
+---------+-----------+----------+------------+--------+----------+-------------+------------+--------------------+
|ORD-00001|   CUST-229|2026-10-07|   Completed|       2|2274478.57|          0.0|  4548957.14|          4548957.14|
|ORD-00001|   CUST-229|2026-10-07|   Completed|       5|1193248.21|          0.0|  5966241.05|          5966241.05|
|ORD-00001|   CUST-229|2026-10-07|   Completed|       1| 698654.31|          0.2|   698654.31|           558923.45|
|ORD-00001|   CUST-229|2026-10-07|   Completed|       3|1373263.87|          0.2|  4119791.61|          3295833.29|
|ORD-00004|   CUST-332|2026-10-12|   Completed|       3|2461096.07|          0.0|  7383288.21|          7383288.21|
|ORD-00008|   CUST-337|2026-10-11|   Completed|       4| 2436907.1|     

In [ ]:
window_cumulative= (Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow))

cumulative_gmv = (
    df_order_summary
    .withColumn("cumulative_gmv", F.round(F.sum("total_after_discount")
            .over(window_cumulative),2)).orderBy("order_date")
)

cumulative_gmv.show()

26/10/20 06:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/20 06:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/20 06:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/20 06:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/20 06:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/20 06:43:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/20 0

+---------+-----------+----------+------------+--------+----------+-------------+-----------+--------------------+--------------+
| order_id|customer_id|order_date|order_status|quantity|     price|discount_rate|total_price|total_after_discount|cumulative_gmv|
+---------+-----------+----------+------------+--------+----------+-------------+-----------+--------------------+--------------+
|ORD-00013|   CUST-299|2026-10-01|   Completed|       2|1987621.36|          0.0| 3975242.72|          3975242.72|    3975242.72|
|ORD-00013|   CUST-299|2026-10-01|   Completed|       3| 815749.81|          0.1| 2447249.43|          2202524.49|    6177767.21|
|ORD-00013|   CUST-299|2026-10-01|   Completed|       1| 262601.77|          0.1|  262601.77|           236341.59|     6414108.8|
|ORD-00013|   CUST-299|2026-10-01|   Completed|       4| 254968.46|         0.15| 1019873.84|           866892.76|    7281001.56|
|ORD-00022|   CUST-448|2026-10-01|   Completed|       4| 1349501.7|          0.2|  5398006

In [ ]:
output_path = "airflow/data_day_46/transformed_orders.parquet"

df_order_summary.write \
    .mode("overwrite") \
    .parquet(output_path)

In [ ]:
import os

print("Output exists:", os.path.exists(output_path))

df_check = spark.read.parquet(output_path)

df_check.printSchema()
df_check.show(truncate=False)

Output exists: True
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- total_price: double (nullable = true)
 |-- total_after_discount: double (nullable = true)

+---------+-----------+----------+------------+--------+----------+-------------+------------+--------------------+
|order_id |customer_id|order_date|order_status|quantity|price     |discount_rate|total_price |total_after_discount|
+---------+-----------+----------+------------+--------+----------+-------------+------------+--------------------+
|ORD-00001|CUST-229   |2026-10-07|Completed   |2       |2274478.57|0.0          |4548957.14  |4548957.14          |
|ORD-00001|CUST-229   |2026-10-07|Completed   |5       |1193248.21|0.0          |5966241.05  |5966241.05          |
|ORD-0000